# List of Lagrangians


## Setup


### Imports


In [1]:

import re
import sys
from pathlib import Path
from fractions import Fraction

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import Expression, S

from model import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    LORENTZ_INDEX,
    SPINOR_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
    CovD,
    Field,
    FieldStrength,
    Gamma,
    Gamma5,
    GaugeFixing,
    GaugeGroup,
    GaugeRepresentation,
    GhostField,
    GhostLagrangian,
    Model,
    PartialD,
    flavor_index,
)
from symbolic.spenso_structures import (
    gauge_generator,
    structure_constant,
    weak_gauge_generator,
    weak_structure_constant,
)
from symbolic.vertex_engine import I, compact_sum_notation, compact_vertex_sum_form

ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")



def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


def show_model(model, *fields, compact_form=None, sum_notation=None):
    source_terms = model.lagrangian_decl.source_terms
    if source_terms:
        lagrangian_source = sum(source_terms[1:], source_terms[0]) if len(source_terms) > 1 else source_terms[0]
        show("Lagrangian", lagrangian_source)
    lagrangian = model.lagrangian()
    if fields:
        show("Feynman Rule", lagrangian.feynman_rule(*fields, include_delta=False))
    else:
        show("Feynman Rules", lagrangian.feynman_rule(include_delta=False))
    if compact_form is not None:
        show("Compact Form", compact_form)
    if sum_notation is not None:
        show("Sum Notation", sum_notation)


### Symbols


In [2]:
mu, nu, rho = S("mu"), S("nu"), S("rho")
eQED, gS, g2, xiQCD = S("eQED"), S("gS"), S("g2"), S("xiQCD")
qPhi, qPsi, qQ, YL = S("qPhi"), S("qPsi"), S("qQ"), S("YL")
lam, y, g4, gGamma = S("lam"), S("y"), S("g4"), S("gGamma")
lam4, g, lamC, gD2, gijk = S("lam4"), S("g"), S("lamC"), S("gD2"), S("gijk")
yF, gV, gJJ, g_psi4, g1, g2f = S("yF"), S("gV"), S("gJJ"), S("g_psi4"), S("g1"), S("g2f")
i, j, k = S("i"), S("j"), S("k")
YH = S("YH")
xiW = S("xiW")


### Fields


In [3]:

Phi = Field(
    "Phi",
    spin=0,
    self_conjugate=False,
    symbol=S("phi"),
    conjugate_symbol=S("phidag"),
    quantum_numbers={"Q": qPhi},
)
Chi = Field(
    "Chi",
    spin=0,
    self_conjugate=False,
    symbol=S("chi"),
    conjugate_symbol=S("chidag"),
)
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": qPsi},
)
Xi = Field(
    "Xi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("xi"),
    conjugate_symbol=S("xibar"),
    indices=(SPINOR_INDEX,),
)
Quark = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("q"),
    conjugate_symbol=S("qbar"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Q": qQ},
)
Photon = Field("A", spin=1, self_conjugate=True, symbol=S("A0"), indices=(LORENTZ_INDEX,))
Gluon = Field("G", spin=1, self_conjugate=True, symbol=S("G0"), indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX))
GhostG = GhostField(
    "ghG",
    ghost_of=Gluon,
    self_conjugate=False,
    symbol=S("ghG0"),
    conjugate_symbol=S("ghGbar0"),
    indices=(COLOR_ADJ_INDEX,),
    quantum_numbers={"GhostNumber": 1},
)
W = Field("W", spin=1, self_conjugate=True, symbol=S("W0"), indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX))
GhostW = GhostField(
    "ghW",
    ghost_of=W,
    self_conjugate=False,
    symbol=S("ghW0"),
    conjugate_symbol=S("ghWbar0"),
    indices=(WEAK_ADJ_INDEX,),
    quantum_numbers={"GhostNumber": 1},
)
L = Field(
    "L",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("L0"),
    conjugate_symbol=S("Lbar0"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX),
    quantum_numbers={"Y": YL},
)

B = Field(
    "B",
    spin=1,
    self_conjugate=True,
    symbol=S("B0"),
    indices=(LORENTZ_INDEX,),
)
HY = Field(
    "HY",
    spin=0,
    self_conjugate=False,
    symbol=S("HY0"),
    conjugate_symbol=S("HYdag0"),
    indices=(WEAK_FUND_INDEX,),
    quantum_numbers={"Y": YH},
)


# Compact scalar-only species for the short model-layer examples below.
PhiR = Field("PhiR", spin=0, self_conjugate=True, symbol=S("phi0"))
ChiR = Field("ChiR", spin=0, self_conjugate=True, symbol=S("chi0"))
PhiC = Field("PhiC", spin=0, self_conjugate=False, symbol=S("phiC0"), conjugate_symbol=S("phiCdag0"))
Phi_i = Field("phi_i", spin=0, self_conjugate=True, symbol=S("phi_i0"))
Phi_j = Field("phi_j", spin=0, self_conjugate=True, symbol=S("phi_j0"))
Phi_k = Field("phi_k", spin=0, self_conjugate=True, symbol=S("phi_k0"))

(Phi, Chi, Psi, Xi, Quark, Photon, Gluon, GhostG, W, GhostW, HY, L)


(Field(name='Phi', spin=0, self_conjugate=False, indices=(), kind='scalar', statistics='boson', symbol=phi, conjugate_symbol=phidag, mass=None, quantum_numbers={'Q': qPhi}, ghost_of=None, flavor_index=None, class_members=()),
 Field(name='Chi', spin=0, self_conjugate=False, indices=(), kind='scalar', statistics='boson', symbol=chi, conjugate_symbol=chidag, mass=None, quantum_numbers={}, ghost_of=None, flavor_index=None, class_members=()),
 Field(name='Psi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', dimension=None, is_flavor=False, prefix='i'),), kind='fermion', statistics='fermion', symbol=psi, conjugate_symbol=psibar, mass=None, quantum_numbers={'Q': qPsi}, ghost_of=None, flavor_index=None, class_members=()),
 Field(name='Xi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4

### Gauge Representations


In [4]:
COLOR_FUND_REP = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fund",
)
WEAK_DOUBLET_REP = GaugeRepresentation(
    index=WEAK_FUND_INDEX,
    generator_builder=weak_gauge_generator,
    name="doublet",
)

(COLOR_FUND_REP, WEAK_DOUBLET_REP)


(GaugeRepresentation(index=IndexType(name='ColorFund', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='color_fund', dimension=None, is_flavor=False, prefix='c'), generator_builder=<function gauge_generator at 0x10cd50eb0>, name='fund', slot=None, slot_policy='unique'),
 GaugeRepresentation(index=IndexType(name='WeakFund', representation=Representation { rep: Dualizable(3), dim: Concrete(2) }, kind='weak_fund', dimension=None, is_flavor=False, prefix='w'), generator_builder=<function weak_gauge_generator at 0x10cd51010>, name='doublet', slot=None, slot_policy='unique'))

### Gauge Groups


In [5]:
U1QED = GaugeGroup(
    name="U1QED",
    abelian=True,
    coupling=eQED,
    gauge_boson=Photon,
    charge="Q",
)
U1Y = GaugeGroup(
    name="U1Y",
    abelian=True,
    coupling=g1,
    gauge_boson=B,
    charge="Y",
)
SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=Gluon,
    ghost_field=GhostG,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)
SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W,
    ghost_field=GhostW,
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)

(U1QED, SU3C, SU2L)


(GaugeGroup(name='U1QED', abelian=True, coupling=eQED, gauge_boson=Field(name='A', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=None, is_flavor=False, prefix='mu'),), kind='vector', statistics='boson', symbol=A0, conjugate_symbol=None, mass=None, quantum_numbers={}, ghost_of=None, flavor_index=None, class_members=()), ghost_field=None, structure_constant=None, representations=(), charge='Q'),
 GaugeGroup(name='SU3C', abelian=False, coupling=gS, gauge_boson=Field(name='G', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=None, is_flavor=False, prefix='mu'), IndexType(name='ColorAdj', representation=Representation { rep: SelfDual(2), dim: Concrete(8) }, kind='color_adj', dimension=None, is_flavor=False, prefix='a')), kind='vector', statistics='boson', symbol

## Scalar Examples


In [6]:
phi4 = Model(lam4 * PhiR * PhiR * PhiR * PhiR)
show_model(phi4)

phi2chi2 = Model(g * PhiR * PhiR * ChiR * ChiR)
show_model(phi2chi2)

complex_bilinear = Model(lamC * PhiC.bar * PhiC)
show_model(complex_bilinear)

multi_species = Model(
    gijk(i, j, k) * Phi_i * Phi_i * Phi_j * Phi_j * Phi_k * Phi_k,
)
show_model(multi_species)


Lagrangian
lam4 * PhiR * PhiR * PhiR * PhiR

Feynman Rules
1 vertex signature(s)

Vertex: ('PhiR', 'PhiR', 'PhiR', 'PhiR')
Rule: 24𝑖*lam4

Lagrangian
g * PhiR * PhiR * ChiR * ChiR

Feynman Rules
1 vertex signature(s)

Vertex: ('PhiR', 'PhiR', 'ChiR', 'ChiR')
Rule: 4𝑖*g

Lagrangian
lamC * PhiC.bar * PhiC

Feynman Rules
1 vertex signature(s)

Vertex: ('PhiC.bar', 'PhiC')
Rule: 1𝑖*lamC

Lagrangian
gijk(i,j,k) * phi_i * phi_i * phi_j * phi_j * phi_k * phi_k

Feynman Rules
1 vertex signature(s)

Vertex: ('phi_i', 'phi_i', 'phi_j', 'phi_j', 'phi_k', 'phi_k')
Rule: 8𝑖*gijk(i,j,k)



In [7]:

derivative_phi4 = Model(
    gD2 * PartialD(PhiR, mu) * PartialD(PhiR, mu) * PhiR * PhiR,
)
show_model(derivative_phi4)

Lagrangian
gD2 * PartialD(PhiR, mu) * PartialD(PhiR, mu) * PhiR * PhiR

Feynman Rules
1 vertex signature(s)

Vertex: ('PhiR', 'PhiR', 'PhiR', 'PhiR')
Rule: -4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)-4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q3,mu1_int)-4𝑖*gD2*pcomp(q1,mu1_int)*pcomp(q4,mu1_int)-4𝑖*gD2*pcomp(q2,mu1_int)*pcomp(q3,mu1_int)-4𝑖*gD2*pcomp(q2,mu1_int)*pcomp(q4,mu1_int)-4𝑖*gD2*pcomp(q3,mu1_int)*pcomp(q4,mu1_int)



## Fermion Examples


In [8]:
yukawa_model = Model(yF * Psi.bar * Psi * Phi)
show_model(yukawa_model)

vector_current_model = Model(gV * Psi.bar * Gamma(mu) * Psi * Photon)
show_model(vector_current_model)

axial_current_model = Model(
    gV * Psi.bar * Gamma(mu) * Gamma5() * Psi * Photon,
)
show_model(axial_current_model)

four_fermion_model = Model(
    -(g_psi4 / Expression.num(2)) * Psi.bar * Psi * Psi.bar * Psi,
)
show_model(four_fermion_model)

current_current_model = Model(
    gJJ * Psi.bar * Gamma(mu) * Psi * Psi.bar * Gamma(mu) * Psi,
)
show_model(current_current_model)

dpsibar_model = Model(
    yF * PartialD(Psi.bar, mu) * Psi * Phi * Phi,
)
show_model(dpsibar_model)

dpsi_model = Model(
    yF * Psi.bar * PartialD(Psi, nu) * Phi * Chi,
)
show_model(dpsi_model)

dphi_dchi_model = Model(
    yF * Psi.bar * Psi * PartialD(Phi, mu) * PartialD(Chi, nu),
)
show_model(dphi_dchi_model)

d2phi_chi_model = Model(
    g1 * Psi.bar * Psi * PartialD(PartialD(Phi, mu), mu) * Chi,
)
show_model(d2phi_chi_model)

d2phi2_model = Model(
    (
        g2f
        * Psi.bar
        * Psi
        * PartialD(PartialD(Phi, mu), nu)
        * PartialD(PartialD(Phi, mu), nu)
    ),
)
show_model(d2phi2_model)


Lagrangian
yF * Psi.bar * Psi * Phi

Feynman Rules
1 vertex signature(s)

Vertex: ('Psi.bar', 'Psi', 'Phi')
Rule: 1𝑖*yF*g(bis(4, i1),bis(4, i2))

Lagrangian
gV * Psi.bar * Gamma(mu) * Psi * A

Feynman Rules
1 vertex signature(s)

Vertex: ('Psi.bar', 'Psi', 'A')
Rule: 1𝑖*gV*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

Lagrangian
gV * Psi.bar * Gamma(mu) * Gamma5() * Psi * A

Feynman Rules
1 vertex signature(s)

Vertex: ('Psi.bar', 'Psi', 'A')
Rule: 1𝑖*gV*gamma(bis(4, i1),bis(4, spinor_decl_3),mink(4, mu3))*gamma5(bis(4, spinor_decl_3),bis(4, i2))

Lagrangian
-1/2*g_psi4 * Psi.bar * Psi * Psi.bar * Psi

Feynman Rules
1 vertex signature(s)

Vertex: ('Psi.bar', 'Psi', 'Psi.bar', 'Psi')
Rule: -1𝑖*g_psi4*g(bis(4, i1),bis(4, i2))*g(bis(4, i3),bis(4, i4))-1𝑖*g_psi4*g(bis(4, i1),bis(4, i4))*g(bis(4, i2),bis(4, i3))

Lagrangian
gJJ * Psi.bar * Gamma(mu) * Psi * Psi.bar * Gamma(mu) * Psi

Feynman Rules
1 vertex signature(s)

Vertex: ('Psi.bar', 'Psi', 'Psi.bar', 'Psi')
Rule: 2𝑖*gJJ*gamma(bis(4, i1)

In [9]:
l1=Model(lam * Psi.bar * Psi* Psi.bar * Psi)
l2=Model(lam * Psi(i) * Psi.bar(i) * Psi(j) * Psi.bar(j))
l3=Model(lam * Psi(i) * Psi.bar(i) * Psi.bar(j) * Psi(j))
show_model(l1)
show_model(l2)
show_model(l3)

Lagrangian
lam * Psi.bar * Psi * Psi.bar * Psi

Feynman Rules
1 vertex signature(s)

Vertex: ('Psi.bar', 'Psi', 'Psi.bar', 'Psi')
Rule: 2𝑖*lam*g(bis(4, i1),bis(4, i2))*g(bis(4, i3),bis(4, i4))+2𝑖*lam*g(bis(4, i1),bis(4, i4))*g(bis(4, i2),bis(4, i3))

Lagrangian
lam * Psi * Psi.bar * Psi * Psi.bar

Feynman Rules
1 vertex signature(s)

Vertex: ('Psi', 'Psi.bar', 'Psi', 'Psi.bar')
Rule: 2𝑖*lam*g(bis(4, i1),bis(4, i2))*g(bis(4, i3),bis(4, i4))+2𝑖*lam*g(bis(4, i1),bis(4, i4))*g(bis(4, i2),bis(4, i3))

Lagrangian
lam * Psi * Psi.bar * Psi.bar * Psi

Feynman Rules
1 vertex signature(s)

Vertex: ('Psi', 'Psi.bar', 'Psi.bar', 'Psi')
Rule: 2𝑖*lam*g(bis(4, i1),bis(4, i2))*g(bis(4, i3),bis(4, i4))+2𝑖*lam*g(bis(4, i1),bis(4, i3))*g(bis(4, i2),bis(4, i4))



## Gauge Examples


In [10]:

qed_fermion_model = Model(
    I * Psi.bar * Gamma(mu) * CovD(Psi, mu),
    gauge_groups=(U1QED,),
)
show_model(qed_fermion_model, Psi.bar, Psi, Photon)

qcd_fermion_model = Model(
    I * Quark.bar * Gamma(mu) * CovD(Quark, mu),
    gauge_groups=(SU3C,),
)
show_model(qcd_fermion_model, Quark.bar, Quark, Gluon)

scalar_qed_model = Model(
    CovD(Phi.bar, mu) * CovD(Phi, mu),
    gauge_groups=(U1QED,),
)
show_model(scalar_qed_model, Phi.bar, Phi, Photon)
show_model(scalar_qed_model, Phi.bar, Phi, Photon, Photon)

photon_kinetic_model = Model(
    -(Expression.num(1) / Expression.num(4)) * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, mu, nu),
)
show_model(photon_kinetic_model, Photon, Photon)

gluon_kinetic_model = Model(
    -(Expression.num(1) / Expression.num(4)) * FieldStrength(SU3C, mu, nu, S("aC")) * FieldStrength(SU3C, mu, nu, S("aC")),
)
show_model(gluon_kinetic_model, Gluon, Gluon, Gluon)

gauge_fixing_model = Model(
    GaugeFixing(U1QED, xi=S("xi")),
)
show_model(gauge_fixing_model, Photon, Photon)

ghost_model = Model(
    GhostLagrangian(SU3C),
)
show_model(ghost_model, GhostG.bar, Gluon, GhostG)

qcd_ghost_model_manual = Model(
    -(GhostG.bar * PartialD(CovD(GhostG, mu), mu)),
    gauge_groups=(SU3C,),
)
show_model(qcd_ghost_model_manual, GhostG.bar, GhostG)
show_model(qcd_ghost_model_manual, GhostG.bar, Gluon, GhostG)

Lagrangian
1𝑖 * Psi.bar * Gamma(mu) * CovD(Psi, mu)

Feynman Rule
1𝑖*eQED*qPsi*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

Lagrangian
1𝑖 * q.bar * Gamma(mu) * CovD(q, mu)

Feynman Rule
1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))

Lagrangian
CovD(Phi.bar, mu) * CovD(Phi, mu)

Feynman Rule
-1𝑖*eQED*qPhi*pcomp(q1,mu)+1𝑖*eQED*qPhi*pcomp(q2,mu)

Lagrangian
CovD(Phi.bar, mu) * CovD(Phi, mu)

Feynman Rule
2𝑖*eQED^2*qPhi^2*g(mink(4, mu3),mink(4, mu4))

Lagrangian
-1/4 * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, mu, nu)

Feynman Rule
-1𝑖*pcomp(q1,mu2)*pcomp(q2,mu1)+1𝑖*g(mink(4, mu1),mink(4, mu2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)

Lagrangian
-1/4 * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC)

Feynman Rule
-gS*g(mink(4, mu3),mink(4, mu1))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q1,mu2)+gS*g(mink(4, mu3),mink(4, mu1))*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q3,mu2)+gS*g(mink(4, mu3),mink(4, mu2))*f(coad(8, a3),coa

In [11]:

xiQED = S("xiQED")

qed_gf_helper = Model(
    GaugeFixing(U1QED, xi=xiQED),
)
qed_gf_manual = Model(
    (
        -(Expression.num(1) / (Expression.num(2) * xiQED))
        * PartialD(Photon(S("mu_qed_gf")), S("mu_qed_gf"))
        * PartialD(Photon(S("nu_qed_gf")), S("nu_qed_gf"))
    ),
)

show(
    "QED Gauge-Fixing Helper Rule",
    qed_gf_helper.lagrangian().feynman_rule(Photon, Photon, include_delta=False),
)
show(
    "QED Gauge-Fixing Manual Rule",
    qed_gf_manual.lagrangian().feynman_rule(Photon, Photon, include_delta=False),
)

qcd_gf_helper = Model(
    GaugeFixing(SU3C, xi=xiQCD),
)
qcd_gf_manual = Model(
    (
        -(Expression.num(1) / (Expression.num(2) * xiQCD))
        * PartialD(Gluon(S("mu_qcd_gf"), S("a_qcd_gf")), S("mu_qcd_gf"))
        * PartialD(Gluon(S("nu_qcd_gf"), S("a_qcd_gf")), S("nu_qcd_gf"))
    ),
)

show(
    "QCD Gauge-Fixing Helper Rule",
    qcd_gf_helper.lagrangian().feynman_rule(Gluon, Gluon, include_delta=False),
)
show(
    "QCD Gauge-Fixing Manual Rule",
    qcd_gf_manual.lagrangian().feynman_rule(Gluon, Gluon, include_delta=False),
)

qcd_gf_ghost_direct = Model(
    GaugeFixing(SU3C, xi=xiQCD) - GhostG.bar * PartialD(CovD(GhostG, mu), mu),
)
show(
    "QCD Gauge-Fixing + Ghost Rules",
    qcd_gf_ghost_direct.lagrangian().feynman_rule(include_delta=False),
)


QED Gauge-Fixing Helper Rule
1𝑖*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQED

QED Gauge-Fixing Manual Rule
1𝑖*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQED

QCD Gauge-Fixing Helper Rule
1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQCD

QCD Gauge-Fixing Manual Rule
1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQCD

QCD Gauge-Fixing + Ghost Rules
3 vertex signature(s)

Vertex: ('G', 'G')
Rule: 1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1)*pcomp(q2,mu2)/xiQCD

Vertex: ('ghG.bar', 'ghG')
Rule: 1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q2,mu1_int)^2

Vertex: ('ghG.bar', 'ghG', 'G')
Rule: gS*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q2,mu3)+gS*f(coad(8, a3),coad(8, a1),coad(8, a2))*pcomp(q3,mu3)



## SU(2) Examples


In [12]:
su2_fermion_model = Model(
    I * L.bar * Gamma(mu) * CovD(L, mu),
    gauge_groups=(SU2L,),
)
show_model(su2_fermion_model)

su2_scalar_model = Model(
    CovD(HY.bar, mu) * CovD(HY, mu),
    gauge_groups=(SU2L,),
)
show_model(su2_scalar_model)

su2_ym_model = Model(
    -(Expression.num(1) / Expression.num(4)) * FieldStrength(SU2L, mu, nu, S("aW")) * FieldStrength(SU2L, mu, nu, S("aW")),
)
show_model(su2_ym_model)

su2_gf_ghost_model = Model(
    GaugeFixing(SU2L, xi=xiW) + GhostLagrangian(SU2L),
)
show_model(su2_gf_ghost_model)

ew_fermion_model = Model(
    I * L.bar * Gamma(mu) * CovD(L, mu),
    gauge_groups=(SU2L, U1Y),
)
show_model(ew_fermion_model)

ew_scalar_model = Model(
    CovD(HY.bar, mu) * CovD(HY, mu),
    gauge_groups=(SU2L, U1Y),
)
show_model(ew_scalar_model)


Lagrangian
1𝑖 * L.bar * Gamma(mu) * CovD(L, mu)

Feynman Rules
2 vertex signature(s)

Vertex: ('L.bar', 'L', 'W')
Rule: 1𝑖*g2*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(3, aw3),cof(2, w1),cof(2, w2))

Vertex: ('L.bar', 'L')
Rule: 1𝑖*g(cof(2, w1),cof(2, w2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu1_int))*pcomp(q2,mu1_int)

Lagrangian
CovD(HY.bar, mu) * CovD(HY, mu)

Feynman Rules
3 vertex signature(s)

Vertex: ('HY.bar', 'HY', 'W')
Rule: -1𝑖*g2*t(coad(3, aw3),cof(2, w1),cof(2, w2))*pcomp(q1,mu)+1𝑖*g2*t(coad(3, aw3),cof(2, w1),cof(2, w2))*pcomp(q2,mu)

Vertex: ('HY.bar', 'HY', 'W', 'W')
Rule: 1𝑖*g2^2*g(mink(4, mu3),mink(4, mu4))*t(coad(3, aw3),cof(2, w1),cof(2, w_mid_HY_SU2L))*t(coad(3, aw4),cof(2, w_mid_HY_SU2L),cof(2, w2))+1𝑖*g2^2*g(mink(4, mu3),mink(4, mu4))*t(coad(3, aw3),cof(2, w_mid_HY_SU2L),cof(2, w2))*t(coad(3, aw4),cof(2, w1),cof(2, w_mid_HY_SU2L))

Vertex: ('HY.bar', 'HY')
Rule: -1𝑖*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)

Lagrangian
-1/4 * FieldStreng

## Standard Model Gauge Structure

We now keep the unbroken Standard Model gauge sector but declare the fermions as three-generation flavor classes.
The gauge interactions stay generation blind, so `flavor_expand=True` makes the family labels explicit and exposes the flavor metric that enforces diagonal gauge couplings.


In [13]:
def Q(n: int) -> Expression:
    return Expression.num(n)


Y_Q_L = Q(1) / Q(6)
Y_u_R = Q(2) / Q(3)
Y_d_R = -(Q(1) / Q(3))
Y_L_L = -(Q(1) / Q(2))
Y_e_R = -Q(1)
Y_H = Q(1) / Q(2)
Generation = flavor_index("Generation", 3, prefix="f")


# QL[f] = (u_L, d_L), (c_L, s_L), (t_L, b_L)
# LL[f] = (nu_eL, e_L), (nu_muL, mu_L), (nu_tauL, tau_L)
QL = Field(
    "QL",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("QL0"),
    conjugate_symbol=S("QLbar0"),
    indices=(Generation, SPINOR_INDEX, COLOR_FUND_INDEX, WEAK_FUND_INDEX),
    quantum_numbers={"Y": Y_Q_L},
    flavor_index=Generation,
    class_members=("Q1L", "Q2L", "Q3L"),
)
Q1L, Q2L, Q3L = QL.class_members

UR = Field(
    "UR",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("UR0"),
    conjugate_symbol=S("URbar0"),
    indices=(Generation, SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Y": Y_u_R},
    flavor_index=Generation,
    class_members=("uR", "cR", "tR"),
)
uR_member, cR, tR = UR.class_members

DR = Field(
    "DR",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("DR0"),
    conjugate_symbol=S("DRbar0"),
    indices=(Generation, SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Y": Y_d_R},
    flavor_index=Generation,
    class_members=("dR", "sR", "bR"),
)
dR_member, sR, bR = DR.class_members

LL = Field(
    "LL",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("LL0"),
    conjugate_symbol=S("LLbar0"),
    indices=(Generation, SPINOR_INDEX, WEAK_FUND_INDEX),
    quantum_numbers={"Y": Y_L_L},
    flavor_index=Generation,
    class_members=("L1L", "L2L", "L3L"),
)
L1L, L2L, L3L = LL.class_members

ER = Field(
    "ER",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("ER0"),
    conjugate_symbol=S("ERbar0"),
    indices=(Generation, SPINOR_INDEX),
    quantum_numbers={"Y": Y_e_R},
    flavor_index=Generation,
    class_members=("eR", "muR", "tauR"),
)
eR_member, muR, tauR = ER.class_members

H = Field(
    "H",
    spin=0,
    self_conjugate=False,
    symbol=S("HSM0"),
    conjugate_symbol=S("HSMdag0"),
    indices=(WEAK_FUND_INDEX,),
    quantum_numbers={"Y": Y_H},
)

# Keep the compact one-family helper fields used in the Yukawa cells below.
qL_1fam = Field(
    "qL_1fam",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("qL0"),
    conjugate_symbol=S("qLbar0"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX, WEAK_FUND_INDEX),
    quantum_numbers={"Y": Y_Q_L},
)
uR_1fam = Field(
    "uR_1fam",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("uR0"),
    conjugate_symbol=S("uRbar0"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Y": Y_u_R},
)
dR_1fam = Field(
    "dR_1fam",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("dR0"),
    conjugate_symbol=S("dRbar0"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Y": Y_d_R},
)
lL_1fam = Field(
    "lL_1fam",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("lL0"),
    conjugate_symbol=S("lLbar0"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX),
    quantum_numbers={"Y": Y_L_L},
)
eR_1fam = Field(
    "eR_1fam",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("eR0"),
    conjugate_symbol=S("eRbar0"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Y": Y_e_R},
)


In [14]:
sm_full_gauge_model = Model(
    (
        I * QL.bar * Gamma(mu) * CovD(QL, mu)
        + I * UR.bar * Gamma(mu) * CovD(UR, mu)
        + I * DR.bar * Gamma(mu) * CovD(DR, mu)
        + I * LL.bar * Gamma(mu) * CovD(LL, mu)
        + I * ER.bar * Gamma(mu) * CovD(ER, mu)
        + CovD(H.bar, mu) * CovD(H, mu)
        - (Expression.num(1) / Expression.num(4)) * FieldStrength(SU3C, mu, nu, S("aC")) * FieldStrength(SU3C, mu, nu, S("aC"))
        - (Expression.num(1) / Expression.num(4)) * FieldStrength(SU2L, mu, nu, S("aW")) * FieldStrength(SU2L, mu, nu, S("aW"))
        - (Expression.num(1) / Expression.num(4)) * FieldStrength(U1Y, mu, nu) * FieldStrength(U1Y, mu, nu)
    ),
    name="Flavored unbroken SM gauge sector",
)

sm_full_gauge_terms = sm_full_gauge_model.lagrangian_decl.source_terms
show("Lagrangian", sum(sm_full_gauge_terms[1:], sm_full_gauge_terms[0]))

print("Flavor classes:")
print("  QL:", [member.name for member in QL.class_members], "-> (u, d)_L, (c, s)_L, (t, b)_L")
print("  UR:", [member.name for member in UR.class_members])
print("  DR:", [member.name for member in DR.class_members])
print("  LL:", [member.name for member in LL.class_members], "-> (nu_e, e)_L, (nu_mu, mu)_L, (nu_tau, tau)_L")
print("  ER:", [member.name for member in ER.class_members])
print()

compact_signatures = [signature.names for signature in sm_full_gauge_model.lagrangian().vertex_signatures()]
expanded_signatures = [signature.names for signature in sm_full_gauge_model.lagrangian().vertex_signatures(flavor_expand=True)]
print("compact signature count =", len(compact_signatures))
print("expanded signature count =", len(expanded_signatures))
print()
print("Expanded QL-B signatures:")
for names in expanded_signatures:
    if len(names) == 3 and names[2] == "B" and names[0] in {"Q1L.bar", "Q2L.bar", "Q3L.bar"}:
        print(names)


Lagrangian
1𝑖 * QL.bar * Gamma(mu) * CovD(QL, mu) + 1𝑖 * UR.bar * Gamma(mu) * CovD(UR, mu) + 1𝑖 * DR.bar * Gamma(mu) * CovD(DR, mu) + 1𝑖 * LL.bar * Gamma(mu) * CovD(LL, mu) + 1𝑖 * ER.bar * Gamma(mu) * CovD(ER, mu) + CovD(H.bar, mu) * CovD(H, mu) + -1/4 * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC) + -1/4 * FieldStrength(SU2L, mu, nu, aW) * FieldStrength(SU2L, mu, nu, aW) + -1/4 * FieldStrength(U1Y, mu, nu) * FieldStrength(U1Y, mu, nu)

Flavor classes:
  QL: ['Q1L', 'Q2L', 'Q3L'] -> (u, d)_L, (c, s)_L, (t, b)_L
  UR: ['uR', 'cR', 'tR']
  DR: ['dR', 'sR', 'bR']
  LL: ['L1L', 'L2L', 'L3L'] -> (nu_e, e)_L, (nu_mu, mu)_L, (nu_tau, tau)_L
  ER: ['eR', 'muR', 'tauR']

compact signature count = 28
expanded signature count = 148

Expanded QL-B signatures:
('Q1L.bar', 'Q1L', 'B')
('Q1L.bar', 'Q2L', 'B')
('Q1L.bar', 'Q3L', 'B')
('Q2L.bar', 'Q1L', 'B')
('Q2L.bar', 'Q2L', 'B')
('Q2L.bar', 'Q3L', 'B')
('Q3L.bar', 'Q1L', 'B')
('Q3L.bar', 'Q2L', 'B')
('Q3L.bar', 'Q3L', 'B')


In [15]:
show(
    "Flavor-expanded rule: Q3L.bar, Q3L, B",
    sm_full_gauge_model.lagrangian().feynman_rule(Q3L.bar, Q3L, B, flavor_expand=True, include_delta=False),
)
show(
    "Flavor-expanded rule: Q1L.bar, Q2L, B",
    sm_full_gauge_model.lagrangian().feynman_rule(Q1L.bar, Q2L, B, flavor_expand=True, include_delta=False),
)
show(
    "Flavor-expanded rule: muR.bar, muR, B",
    sm_full_gauge_model.lagrangian().feynman_rule(muR.bar, muR, B, flavor_expand=True, include_delta=False),
)


Flavor-expanded rule: Q3L.bar, Q3L, B
1𝑖/6*g1*g(cof(3, 3),cof(3, 3))*g(cof(2, w1),cof(2, w2))*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

Flavor-expanded rule: Q1L.bar, Q2L, B
1𝑖/6*g1*g(cof(3, 1),cof(3, 2))*g(cof(2, w1),cof(2, w2))*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

Flavor-expanded rule: muR.bar, muR, B
-1𝑖*g1*g(cof(3, 2),cof(3, 2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))



## Unbroken SM Yukawa Interactions


In [16]:
yu, yd, ye = S("yu"), S("yd"), S("ye")

from symbolic.spenso_structures import weak_eps2 as eps2

i_qu = S("i_qu")
j_qu = S("j_qu")

sm_yukawa_model = Model(
    (
        -yd * qL_1fam.bar * H * dR_1fam
        - yd * dR_1fam.bar * H.bar * qL_1fam
        - ye * lL_1fam.bar * H * eR_1fam
        - ye * eR_1fam.bar * H.bar * lL_1fam
        - yu * eps2(i_qu, j_qu) * qL_1fam.bar(index_labels={WEAK_FUND_INDEX.kind: i_qu}) * H.bar(j_qu) * uR_1fam
        - yu * eps2(i_qu, j_qu) * uR_1fam.bar * H(j_qu) * qL_1fam(index_labels={WEAK_FUND_INDEX.kind: i_qu})
    ),
)


def occ_name(occ):
    if occ.conjugated and not occ.field.self_conjugate:
        return f"{occ.field.name}.bar"
    return occ.field.name


show("Lagrangian", sum(sm_yukawa_model.lagrangian_decl.source_terms[1:], sm_yukawa_model.lagrangian_decl.source_terms[0]))

print("==========")
print("Lowered Terms")
for idx, term in enumerate(sm_yukawa_model.lagrangian().terms, 1):
    print(f"Term {idx}:")
    print("coupling =", clean(term.coupling))
    for field_idx, occ in enumerate(term.fields, 1):
        print(f"field {field_idx} = {occ_name(occ)}, labels = {occ.labels}")
    print()

print("==========")
print("Vertex Signatures")
for signature in sm_yukawa_model.lagrangian().vertex_signatures():
    print(signature.names)
print()

show("Feynman Rules", sm_yukawa_model.lagrangian().feynman_rule(include_delta=False))


Lagrangian
-yd * qL_1fam.bar * H * dR_1fam + -yd * dR_1fam.bar * H.bar * qL_1fam + -ye * lL_1fam.bar * H * eR_1fam + -ye * eR_1fam.bar * H.bar * lL_1fam + -yu*weak_eps2(cof(2, i_qu),cof(2, j_qu)) * qL_1fam.bar * H.bar * uR_1fam + -yu*weak_eps2(cof(2, i_qu),cof(2, j_qu)) * uR_1fam.bar * H * qL_1fam

Lowered Terms
Term 1:
coupling = -yd
field 1 = qL_1fam.bar, labels = {'spinor': i_decl_1, 'color_fund': c_decl_1, 'weak_fund': w_decl_1}
field 2 = H, labels = {'weak_fund': w_decl_1}
field 3 = dR_1fam, labels = {'spinor': i_decl_1, 'color_fund': c_decl_1}

Term 2:
coupling = -yd
field 1 = dR_1fam.bar, labels = {'spinor': i_decl_1, 'color_fund': c_decl_1}
field 2 = H.bar, labels = {'weak_fund': w_decl_1}
field 3 = qL_1fam, labels = {'spinor': i_decl_1, 'color_fund': c_decl_1, 'weak_fund': w_decl_1}

Term 3:
coupling = -ye
field 1 = lL_1fam.bar, labels = {'spinor': i_decl_1, 'weak_fund': w_decl_1}
field 2 = H, labels = {'weak_fund': w_decl_1}
field 3 = eR_1fam, labels = {'spinor': i_decl_1}

T

## Unbroken Higgs Potential


In [17]:
muH2 = S("muH2")
lamH = S("lamH")

sm_higgs_potential_model = Model(
    muH2 * H.bar * H - lamH * (H.bar * H) * (H.bar * H),
)

show_model(sm_higgs_potential_model)


Lagrangian
muH2 * H.bar * H + -lamH * H.bar * H * H.bar * H

Feynman Rules
2 vertex signature(s)

Vertex: ('H.bar', 'H')
Rule: 1𝑖*muH2*g(cof(2, w1),cof(2, w2))

Vertex: ('H.bar', 'H', 'H.bar', 'H')
Rule: -2𝑖*lamH*g(cof(2, w1),cof(2, w2))*g(cof(2, w3),cof(2, w4))-2𝑖*lamH*g(cof(2, w1),cof(2, w4))*g(cof(2, w2),cof(2, w3))



# test to ignore 

### Weak-Index Contraction Diagnostic


In [18]:
def occ_text(occ):
    label = occ.labels.get("weak_fund")
    idx = f"[{label}]" if label is not None else ""
    return f"{occ.field.name}.bar{idx}" if occ.conjugated else f"{occ.field.name}{idx}"

def compiled_term_text(term):
    return f"{clean(term.coupling)} * " + " * ".join(occ_text(occ) for occ in term.fields)

def canon(expr):
    return expr.expand().to_canonical_string()

compact_higgs_terms = sm_higgs_potential_model.lagrangian_decl.source_terms
compact_higgs_decl = sum(compact_higgs_terms[1:], compact_higgs_terms[0]) if len(compact_higgs_terms) > 1 else compact_higgs_terms[0]

iH, jH = S("iH"), S("jH")
explicit_higgs_potential_decl = (
    muH2 * H.bar(iH) * H(iH)
    - lamH * H.bar(iH) * H(iH) * H.bar(jH) * H(jH)
)
explicit_higgs_potential_model = Model(
    explicit_higgs_potential_decl,
)

show("Raw compact lagrangian_decl", compact_higgs_decl)
print("Lowered compact terms:")
for term in sm_higgs_potential_model.lagrangian().terms:
    print(compiled_term_text(term))
print()

print("Lowered explicit-index terms:")
for term in explicit_higgs_potential_model.lagrangian().terms:
    print(compiled_term_text(term))
print()

print("Quadratic Higgs term with explicit weak indices:")
print(compiled_term_text(explicit_higgs_potential_model.lagrangian().terms[0]))
print()
print("Quartic Higgs term with explicit weak indices:")
print(compiled_term_text(explicit_higgs_potential_model.lagrangian().terms[1]))
print()

show("Compact Higgs-potential rules", sm_higgs_potential_model.lagrangian().feynman_rule(include_delta=False))
show("Explicit-index Higgs-potential rules", explicit_higgs_potential_model.lagrangian().feynman_rule(include_delta=False))

compact_higgs_signatures = tuple(sig.names for sig in sm_higgs_potential_model.lagrangian().vertex_signatures())
explicit_higgs_signatures = tuple(sig.names for sig in explicit_higgs_potential_model.lagrangian().vertex_signatures())
compact_hh_rule = sm_higgs_potential_model.lagrangian().feynman_rule(H.bar, H, include_delta=False)
explicit_hh_rule = explicit_higgs_potential_model.lagrangian().feynman_rule(H.bar, H, include_delta=False)
compact_hhhh_rule = sm_higgs_potential_model.lagrangian().feynman_rule(H.bar, H, H.bar, H, include_delta=False)
explicit_hhhh_rule = explicit_higgs_potential_model.lagrangian().feynman_rule(H.bar, H, H.bar, H, include_delta=False)

print("same signatures =", compact_higgs_signatures == explicit_higgs_signatures)
print("same H.bar-H rule =", canon(compact_hh_rule) == canon(explicit_hh_rule))
print("same H.bar-H-H.bar-H rule =", canon(compact_hhhh_rule) == canon(explicit_hhhh_rule))
print("compiled-lagrangian direct subtraction = not supported cleanly; comparing emitted rules instead")
print()
show("2-point rule difference", (compact_hh_rule - explicit_hh_rule).expand())
show("4-point rule difference", (compact_hhhh_rule - explicit_hhhh_rule).expand())


Raw compact lagrangian_decl
muH2 * H.bar * H + -lamH * H.bar * H * H.bar * H

Lowered compact terms:
muH2 * H.bar[w_decl_1] * H[w_decl_1]
-lamH * H.bar[w_decl_1] * H[w_decl_1] * H.bar[w_decl_2] * H[w_decl_2]

Lowered explicit-index terms:
muH2 * H.bar[iH] * H[iH]
-lamH * H.bar[iH] * H[iH] * H.bar[jH] * H[jH]

Quadratic Higgs term with explicit weak indices:
muH2 * H.bar[iH] * H[iH]

Quartic Higgs term with explicit weak indices:
-lamH * H.bar[iH] * H[iH] * H.bar[jH] * H[jH]

Compact Higgs-potential rules
2 vertex signature(s)

Vertex: ('H.bar', 'H')
Rule: 1𝑖*muH2*g(cof(2, w1),cof(2, w2))

Vertex: ('H.bar', 'H', 'H.bar', 'H')
Rule: -2𝑖*lamH*g(cof(2, w1),cof(2, w2))*g(cof(2, w3),cof(2, w4))-2𝑖*lamH*g(cof(2, w1),cof(2, w4))*g(cof(2, w2),cof(2, w3))

Explicit-index Higgs-potential rules
2 vertex signature(s)

Vertex: ('H.bar', 'H')
Rule: 1𝑖*muH2*g(cof(2, w1),cof(2, w2))

Vertex: ('H.bar', 'H', 'H.bar', 'H')
Rule: -2𝑖*lamH*g(cof(2, w1),cof(2, w2))*g(cof(2, w3),cof(2, w4))-2𝑖*lamH*g(cof(2, w